In [1]:
import os
import sys
import zipfile
import re
import csv
import pyarrow.csv as pv
import pyarrow.parquet as pq
import pyarrow as pa
import polars as pl
from warnings import filterwarnings

filterwarnings("ignore")
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [ ]:
#=============================================================================================================
#===================================
# env loader - connection setup
#===================================

current_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(current_dir, "../../")))

from pipelines.commons.env_loader import (
    MINIO_ACCESS_KEY, MINIO_SECRET_KEY, MINIO_ENDPOINT, validate_env,
)
from pipelines.commons.s3_client import get_s3_client
from pipelines.commons.logger import get_logger

logger = get_logger("rfb_extract")

validate_env({
    "MINIO_ACCESS_KEY": MINIO_ACCESS_KEY,
    "MINIO_SECRET_KEY": MINIO_SECRET_KEY,
})

# Configurações do seu S3/MinIO local
s3_lake = {
    "endpoint_url": MINIO_ENDPOINT,
    "aws_access_key_id": MINIO_ACCESS_KEY,
    "aws_secret_access_key": MINIO_SECRET_KEY,
    "region_name": "us-east-1"
}

In [3]:
#=================================================
# s3 reader - bucket bronze
#=================================================
def bucket_bronze_loader(entidade: str, referencia_mes: str, linhas: int = 100000) -> pl.DataFrame:
    caminho_s3 = f"s3://bronze/rfb/{entidade}/ref_month={referencia_mes}/*.parquet"
    print(f"Lendo amostra de: {caminho_s3}...")
    
    try:

        df = pl.scan_parquet(
            caminho_s3,
            storage_options=s3_lake
        ).head(linhas).collect()
        
        return df
    
    except Exception as e:
        print(f"Erro ao carregar dados: {e}")
        return pl.DataFrame()



In [4]:
#=================================================
# s3 reader - bucket silver
#=================================================

def bucket_silver_loader(entidade: str, referencia_mes: str, linhas: int = 100000) -> pl.DataFrame:
    caminho_s3 = f"s3://silver/rfb/{entidade}/ref_month={referencia_mes}/*.parquet"
    print(f"Lendo amostra de: {caminho_s3}...")
    
    try:

        df = pl.scan_parquet(
            caminho_s3,
            storage_options=s3_lake
        ).head(linhas).collect()
        
        return df
    
    except Exception as e:
        print(f"Erro ao carregar dados: {e}")
        return pl.DataFrame()



In [ ]:
# ==========================================
# bronzer loader
# ==========================================

#parquet estabelecimentos
bronze_estabelecimentos = bucket_bronze_loader("estabelecimentos", "202608")
print('bronze_estabelecimentos loaded')
print(f'{bronze_estabelecimentos.shape[0]} rows and {bronze_estabelecimentos.shape[1]} columns ')

# bronze_empresas = bucket_bronze_loader("empresas", "202608")
# print('bronze_empresas loaded')
# print(f'{bronze_empresas.shape[0]} rows and {bronze_empresas.shape[1]} columns ')

# bronze_socios = bucket_bronze_loader("socios", "202608")
# print('bronze_socios loaded')
# print(f'{bronze_socios.shape[0]} rows and {bronze_socios.shape[1]} columns ')

# bronze_simples = bucket_bronze_loader("simples", "202608")
# print('bronze_simples loaded')
# print(f'{bronze_simples.shape[0]} rows and {bronze_simples.shape[1]} columns ')

Lendo amostra de: s3://bronze/rfb/estabelecimentos/ref_month=202608/*.parquet...
bronze_estabelecimentos loaded
100000 rows and 31 columns 


In [ ]:
# ==========================================
# silver loader
# ==========================================
#parquet estabelecimentos
silver_estabelecimentos = bucket_silver_loader("estabelecimentos", "202608")
print('silver_estabelecimentos loaded')
print(f'{silver_estabelecimentos.shape[0]} rows and {silver_estabelecimentos.shape[1]} columns ')

# silver_empresas = bucket_silver_loader("empresas", "202608")
# print('silver_empresas loaded')
# print(f'{silver_empresas.shape[0]} rows and {silver_empresas.shape[1]} columns ')

# silver_socios = bucket_silver_loader("socios", "202608")
# print('silver_socios loaded')
# print(f'{silver_socios.shape[0]} rows and {silver_socios.shape[1]} columns ')

# silver_simples = bucket_silver_loader("simples", "202608")
# print('silver_simples loaded')
# print(f'{silver_simples.shape[0]} rows and {silver_simples.shape[1]} columns ')

Lendo amostra de: s3://silver/rfb/estabelecimentos/ref_month=202608/*.parquet...
silver_estabelecimentos loaded
100000 rows and 34 columns 
